# 17 XGBoost Optuna Prep for E_S

## Import

In [ ]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.metrics import roc_auc_score

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from optuna_integration import XGBoostPruningCallback

from xgboost import XGBClassifier

import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
mpl.style.use("seaborn-v0_8-colorblind")
RANDOM_STATE = 42

## Dataframe

In [ ]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

In [ ]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]
x_full, y_full = df_raw[feat_cols], df_raw["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)],
        "full": [len(df_raw), len(x_full), len(y_full)]
    }
)

## Hilfsvariablen

In [ ]:
calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]
feat_cols_no_calc = [c for c in feat_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]

In [ ]:
def eval_modell(mod_idx, model, x_train, y_train, x_val, y_val, train_time, best_iter=None):
    """AUC auf Train und Val"""
    auc_train = roc_auc_score(y_train, model.predict_proba(x_train)[:, 1])
    auc_val = roc_auc_score(y_val, model.predict_proba(x_val)[:, 1])
    return {
        "model_idx": mod_idx,
        "auc_train": auc_train,
        "auc_val": auc_val,
        "gini": 2 * auc_val - 1,
        "delta_auc": auc_train - auc_val,
        "best_iter": best_iter,
        "trainingszeit": train_time
    }

In [ ]:
x_train_no_calc = x_train[feat_cols_no_calc]
x_val_no_calc = x_val[feat_cols_no_calc]
x_test_no_calc = x_test[feat_cols_no_calc]
x_full_no_calc = x_full[feat_cols_no_calc]

## Optuna Hilfe

## Suchräume

|Paramerter|Bereich|
|---|---|
|learning rate||
|max_depth||
|||
|||
|||
|||
|||
|||
|||

In [ ]:
fixed_params_xgb = {
    "random_state": RANDOM_STATE,
    "verbosity": 0,
    "device": "cpu",
    "n_jobs": -1,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "enable_categorical": True,
    "n_estimators": 3000,
    "early_stopping_rounds": 100
}

In [ ]:
def suchraum_params(trial):
    """Suchraum definition"""
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 200.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 100.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [1, 5, 27]),
        "max_cat_to_onehot": trial.suggest_categorical("max_cat_to_onehot", [2, 10, 18, 105]),
    }

    return params

## Mit Calc

In [ ]:
def objective_xgb_calc(trial):
    pruning_callback = XGBoostPruningCallback(trial, "validation_0-auc")

    modell = XGBClassifier(
        **fixed_params_xgb,
        **suchraum_params(trial),
        callbacks=[pruning_callback]
    )

    modell.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=0)

    return roc_auc_score(y_val, modell.predict_proba(x_val)[:, 1])

In [ ]:
study_xgb_calc = optuna.create_study(
    study_name = "XGBoost_calc",
    storage = "sqlite:///xgboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

In [ ]:
study_xgb_calc.optimize(
    objective_xgb_calc,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

In [ ]:
pd.DataFrame({
    "best auc_val": study_xgb_calc.best_value,
    "best gini": 2 * study_xgb_calc.best_value - 1,
    "trials": len(study_xgb_calc.trials),
    "pruned": len([t for t in study_xgb_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_xgb_calc.trials if t.state.name == "FAIL"]),
    "best params": study_xgb_calc.best_params
})

In [ ]:
study_xgb_calc.best_params

## Ohne Calc

In [ ]:
def objective_xgb_no_calc(trial):
    pruning_callback = XGBoostPruningCallback(trial, "validation_0-auc")

    modell = XGBClassifier(
        **fixed_params_xgb,
        **suchraum_params(trial),
        callbacks=[pruning_callback]
    )

    modell.fit(x_train_no_calc, y_train, eval_set=[(x_val_no_calc, y_val)], verbose=0)

    return roc_auc_score(y_val, modell.predict_proba(x_val_no_calc)[:, 1])

In [ ]:
study_xgb_no_calc = optuna.create_study(
    study_name = "XGBoost_no_calc",
    storage = "sqlite:///xgboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

In [ ]:
study_xgb_no_calc.optimize(
    objective_xgb_no_calc,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

In [ ]:
pd.DataFrame({
    "best auc_val": study_xgb_no_calc.best_value,
    "best gini": 2 * study_xgb_no_calc.best_value - 1,
    "trials": len(study_xgb_no_calc.trials),
    "pruned": len([t for t in study_xgb_no_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_xgb_no_calc.trials if t.state.name == "FAIL"]),
    "best params": study_xgb_no_calc.best_params
})

In [ ]:
study_xgb_no_calc.best_params

## Load Study

In [ ]:
study_xgb_calc_loaded = optuna.load_study(
    study_name = "XGBoost_calc",
    storage = "sqlite:///xgboost_opti.db"
)


In [ ]:
study_xgb_no_calc_loaded = optuna.load_study(
    study_name = "XGBoost_no_calc",
    storage = "sqlite:///xgboost_opti.db"
)

## Best Params again

In [ ]:
study_xgb_calc_loaded.best_params

In [ ]:
study_xgb_no_calc_loaded.best_params

## 100% Train

In [ ]:
results = []
train_times = {}

In [ ]:
name = "L_xgb_opt_01"

L_xgb_opt_01 = XGBClassifier(
    **fixed_params_xgb,
    **study_xgb_calc_loaded.best_params
)

start = time.time()
L_xgb_opt_01.fit(
    x_train, y_train, 
    eval_set=[(x_val, y_val)], 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_xgb_opt_01, x_train, y_train, x_val, y_val, train_times[name], best_iter=L_xgb_opt_01.best_iteration)
)

In [ ]:
name = "L_xgb_opt_02"

L_xgb_opt_02 = XGBClassifier(
    **fixed_params_xgb,
    **study_xgb_no_calc_loaded.best_params
)

start = time.time()
L_xgb_opt_02.fit(
    x_train_no_calc, y_train, 
    eval_set=[(x_val_no_calc, y_val)], 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_xgb_opt_02, x_train_no_calc, y_train, x_val_no_calc, y_val, train_times[name], best_iter=L_xgb_opt_02.best_iteration)
)

In [ ]:
pd.DataFrame(results)

## Save Models

In [ ]:
Modelle = [
    (L_xgb_opt_01, "L_xgb_opt_01"),
    (L_xgb_opt_02, "L_xgb_opt_02")
]

for modell, name in Modelle:
    modell.save_model(f"{name}.json")

## Notizen
- callback in konstruktor nicht in fit wie catboost
- n-warmup etwas erhöht 
- Erics Parameter eingefügt, colsample_bynode verwendet
- max_cat_to_encode mit aufgenommen aus meinen Experimenten zu Catboost


## Ressourcen [Abrufdatum: 18.08.2026]:
- https://optuna.readthedocs.io/en/v3.4.1/reference/generated/optuna.integration.XGBoostPruningCallback.html
- https://optuna-integration.readthedocs.io/en/latest/reference/generated/optuna_integration.XGBoostPruningCallback.html
- https://xgboost.readthedocs.io/en/latest/parameter.html
- https://www.geeksforgeeks.org/machine-learning/xgboost-parameters/
- https://medium.com/optuna/using-optuna-to-optimize-xgboost-hyperparameters-63bfcdfd3407
- https://xgboosting.com/xgboost-hyperparameter-optimization-with-optuna/
- https://www.geeksforgeeks.org/machine-learning/saving-and-loading-xgboost-models/

